# National observed-flood inequality by flood type and population year

This notebook extends the national analysis in `1_observed_figures.ipynb` using the newer country-level GeoPackages. It asks:

1. How does the national concentration index (CI) vary between inland and coastal flooding over the full 2000–2018 observed period?
2. How do country results compare when the same all-period flood surface is evaluated with 2010 population?
3. How does cumulative population exposure change across the 2000, 2005, 2010, 2015, and 2020 population snapshots?

The headline cross-section uses **2010 population**, the central snapshot for the observed period. Negative CI indicates concentration among relatively poorer populations; positive CI indicates concentration among relatively wealthier populations.

## Important interpretation of the population analysis

The `gfd_all` rasters sum flood occurrences through time. Consequently, `Total Flood Risk` is a **frequency-weighted cumulative population exposure** (person-event equivalents), not a count of unique residents inside a binary floodplain. It can exceed total population where locations flooded repeatedly.

Therefore this notebook does **not** calculate outside-floodplain population as `Population - Total Flood Risk`. Instead it compares growth in cumulative exposure within observed flood footprints with growth in the analysed national population. A positive growth gap means cumulative exposure increased faster than population overall; it is consistent with relatively faster population growth in more frequently flooded locations, but it is not an exact binary in-versus-out floodplain estimate.

An exact split requires rerunning the zonal calculation with each `gfd_all` raster converted to a binary mask (`flood frequency > 0`) and returning both population inside and outside that mask.

## 1. Imports and configuration

In [1]:
from pathlib import Path
import re
import sqlite3

import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

RESULTS_RELATIVE_PATH = Path("data/results/social_flood/countries")
HEADLINE_POPULATION_YEAR = 2010
POPULATION_YEARS = [2000, 2005, 2010, 2015, 2020]
FLOOD_TYPES = ["inland", "coastal"]
FLOOD_TYPE_LABELS = {"inland": "Inland", "coastal": "Coastal"}
FLOOD_TYPE_COLORS = {"inland": "#2c7fb8", "coastal": "#d95f0e"}

RWI_COVERAGE_THRESHOLD = 90
EXPOSURE_THRESHOLD = 50_000
EXPOSURE_INTENSITY_THRESHOLD_PERCENT = 2
FOCUS_ISO3 = "KEN"

QUINTILE_COLUMNS = [f"Q{i} Flood Risk" for i in range(1, 6)]

In [ ]:
def find_project_root(start: Path) -> Path:
    """Find the repository root from either the root or notebooks directory."""
    for candidate in (start, *start.parents):
        if (candidate / RESULTS_RELATIVE_PATH).is_dir():
            return candidate
    raise FileNotFoundError(
        f"Could not find {RESULTS_RELATIVE_PATH} above {start.resolve()}."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
COUNTRIES_DIR = PROJECT_ROOT / RESULTS_RELATIVE_PATH

print(f"Project root: {PROJECT_ROOT}")
print(f"Country results: {COUNTRIES_DIR}")
print(f"Headline population year: {HEADLINE_POPULATION_YEAR}")

## 2. Load all-period ADM0 metrics

In [ ]:
FILENAME_PATTERN = re.compile(
    r"^(?P<iso3>[A-Z]{3})_ADM0_metrics_gfd_all_"
    r"(?P<flood_type>inland|coastal)-flood_"
    r"S-(?P<socioeconomic>[^_]+)_P-(?P<population_year>\d{4})\.gpkg$"
)


def parse_result_filename(path: Path) -> dict:
    """Extract dimensions encoded in an all-period result filename."""
    match = FILENAME_PATTERN.match(path.name)
    if match is None:
        raise ValueError(f"Unrecognized all-period result filename: {path.name}")
    metadata = match.groupdict()
    metadata["population_year"] = int(metadata["population_year"])
    metadata["source_path"] = str(path.resolve())
    return metadata


def quote_identifier(name: str) -> str:
    """Safely quote a SQLite identifier."""
    return '"' + name.replace('"', '""') + '"'


def read_adm0_attributes(path: Path) -> dict:
    """Read the single non-spatial ADM0 row directly from a GeoPackage."""
    with sqlite3.connect(path) as connection:
        layers = connection.execute(
            "SELECT table_name FROM gpkg_contents WHERE data_type = 'features'"
        ).fetchall()
        if len(layers) != 1:
            raise ValueError(
                f"Expected one feature layer in {path.name}; found {len(layers)}"
            )

        layer = layers[0][0]
        geometry_rows = connection.execute(
            "SELECT column_name FROM gpkg_geometry_columns WHERE table_name = ?",
            (layer,),
        ).fetchall()
        geometry_columns = {row[0] for row in geometry_rows}
        columns = [
            row[1]
            for row in connection.execute(
                f"PRAGMA table_info({quote_identifier(layer)})"
            )
            if row[1] != "fid" and row[1] not in geometry_columns
        ]
        selected = ", ".join(quote_identifier(column) for column in columns)
        rows = connection.execute(
            f"SELECT {selected} FROM {quote_identifier(layer)}"
        ).fetchall()

    if len(rows) != 1:
        raise ValueError(f"Expected one ADM0 row in {path.name}; found {len(rows)}")
    return dict(zip(columns, rows[0]))


def load_all_period_metrics(countries_dir: Path):
    """Load all recognized all-period ADM0 GeoPackages and a file audit."""
    records = []
    audit = []
    paths = sorted(
        countries_dir.glob(
            "*/inequality_metrics/*_ADM0_metrics_gfd_all_*-flood_S-rwi_P-*.gpkg"
        )
    )
    if not paths:
        raise FileNotFoundError(
            f"No all-period ADM0 metric files found below {countries_dir}"
        )

    for path in paths:
        try:
            records.append(
                {**parse_result_filename(path), **read_adm0_attributes(path)}
            )
            audit.append(
                {"source_path": str(path), "status": "loaded", "detail": ""}
            )
        except Exception as exc:
            audit.append(
                {
                    "source_path": str(path),
                    "status": "failed",
                    "detail": f"{type(exc).__name__}: {exc}",
                }
            )

    if not records:
        raise ValueError("No all-period metric rows could be loaded.")

    metrics = pd.DataFrame.from_records(records)
    audit_frame = pd.DataFrame.from_records(audit)
    numeric_columns = [
        "CI", "QR", "Population", "Population Coverage (%)", "Total Flood Risk",
        *QUINTILE_COLUMNS,
    ]
    for column in numeric_columns:
        metrics[column] = pd.to_numeric(metrics[column], errors="coerce")

    metrics["flood_type_label"] = metrics["flood_type"].map(FLOOD_TYPE_LABELS)
    metrics["has_exposure"] = metrics["Total Flood Risk"].fillna(0).gt(0)
    metrics["inequality_defined"] = metrics["has_exposure"] & metrics["CI"].notna()
    metrics["exposure_intensity"] = np.where(
        metrics["Population"].gt(0),
        metrics["Total Flood Risk"] / metrics["Population"],
        np.nan,
    )
    metrics["quintile_sum"] = metrics[QUINTILE_COLUMNS].sum(axis=1, min_count=1)
    metrics["quintile_sum_difference"] = (
        metrics["quintile_sum"] - metrics["Total Flood Risk"]
    )
    return (
        metrics.sort_values(["iso3", "flood_type", "population_year"]),
        audit_frame,
    )

In [ ]:
all_metrics, file_audit = load_all_period_metrics(COUNTRIES_DIR)

print(
    f"Loaded {len(all_metrics):,} all-period rows for "
    f"{all_metrics['iso3'].nunique()} countries."
)
display(file_audit["status"].value_counts().rename_axis("status").to_frame("files"))
display(
    all_metrics[
        [
            "iso3", "shapeName", "flood_type_label", "population_year",
            "CI", "Population Coverage (%)", "Total Flood Risk",
            "exposure_intensity",
        ]
    ].head(10)
)

## 3. Coverage and consistency checks

In [ ]:
duplicate_keys = ["iso3", "flood_type", "population_year"]
duplicates = all_metrics.duplicated(duplicate_keys, keep=False)

country_coverage = (
    all_metrics.groupby("iso3")
    .agg(
        files=("source_path", "size"),
        flood_types=("flood_type", "nunique"),
        population_snapshots=("population_year", "nunique"),
    )
    .sort_index()
)

population_by_type = all_metrics.pivot_table(
    index=["iso3", "population_year"],
    columns="flood_type",
    values="Population",
    aggfunc="first",
)
population_agrees = np.allclose(
    population_by_type["inland"],
    population_by_type["coastal"],
    equal_nan=True,
    rtol=1e-10,
    atol=1e-6,
)

assert not duplicates.any(), "Duplicate country/flood-type/population-year rows found."
assert set(all_metrics["flood_type"]) == set(FLOOD_TYPES), "Unexpected flood types."
assert set(all_metrics["population_year"]) == set(POPULATION_YEARS), (
    "Population snapshot coverage differs from expectation."
)
assert country_coverage["files"].eq(
    len(FLOOD_TYPES) * len(POPULATION_YEARS)
).all(), "At least one country does not have all ten expected files."
assert population_agrees, "Population differs between inland and coastal files."
assert np.allclose(
    all_metrics["quintile_sum"],
    all_metrics["Total Flood Risk"],
    equal_nan=True,
    rtol=1e-9,
    atol=1e-6,
), "Quintile exposure does not sum to total exposure."

coverage_summary = pd.DataFrame(
    {
        "check": [
            "Countries",
            "Recognized files",
            "Failed files",
            "Countries with complete 2 × 5 coverage",
            "Positive-exposure rows",
            "Rows with defined CI",
            "Minimum RWI population coverage (%)",
            "Maximum absolute quintile-sum difference",
        ],
        "value": [
            all_metrics["iso3"].nunique(),
            len(all_metrics),
            int(file_audit["status"].ne("loaded").sum()),
            int(country_coverage["files"].eq(10).sum()),
            int(all_metrics["has_exposure"].sum()),
            int(all_metrics["inequality_defined"].sum()),
            all_metrics["Population Coverage (%)"].min(),
            all_metrics["quintile_sum_difference"].abs().max(),
        ],
    }
)
display(coverage_summary)
print("Core file coverage and metric consistency checks passed.")

In [ ]:
frequency_weighted_examples = (
    all_metrics.loc[
        all_metrics["exposure_intensity"].gt(1),
        [
            "iso3", "shapeName", "flood_type_label", "population_year",
            "Population", "Total Flood Risk", "exposure_intensity",
        ],
    ]
    .sort_values("exposure_intensity", ascending=False)
)

print(
    f"{len(frequency_weighted_examples)} rows have cumulative exposure greater "
    "than the analysed population. This confirms that Total Flood Risk is "
    "frequency-weighted and cannot be subtracted from population to obtain an "
    "outside-floodplain count."
)
display(frequency_weighted_examples)

## 4. Headline national CI using 2010 population

The original filters are retained: countries require more than 90% population coverage by the RWI surface, positive cumulative exposure, and a defined CI. An asterisk or hatch marks countries with both fewer than 50,000 cumulative person-event exposures and exposure intensity below 2%.

In [ ]:
headline = all_metrics.loc[
    all_metrics["population_year"].eq(HEADLINE_POPULATION_YEAR)
].copy()

headline["uncertain"] = (
    headline["Total Flood Risk"].lt(EXPOSURE_THRESHOLD)
    & headline["exposure_intensity"].mul(100).lt(
        EXPOSURE_INTENSITY_THRESHOLD_PERCENT
    )
)
headline["passes_coverage"] = headline["Population Coverage (%)"].gt(
    RWI_COVERAGE_THRESHOLD
)
headline["valid_for_ci"] = (
    headline["passes_coverage"]
    & headline["has_exposure"]
    & headline["CI"].notna()
)

headline_status = (
    headline.groupby("flood_type_label")
    .agg(
        countries=("iso3", "nunique"),
        positive_exposure=("has_exposure", "sum"),
        defined_CI=("inequality_defined", "sum"),
        passing_all_filters=("valid_for_ci", "sum"),
        uncertain=("uncertain", "sum"),
        median_CI=("CI", "median"),
    )
)
display(headline_status)

### 4.1 National CI maps by flood type

In [ ]:
def load_country_geometries(headline_frame):
    """Read one country geometry per ISO3 from the selected 2010 files."""
    frames = []
    rows = (
        headline_frame.sort_values(["iso3", "flood_type"])
        .drop_duplicates("iso3")
        .itertuples(index=False)
    )
    for row in rows:
        geometry = gpd.read_file(row.source_path)[["geometry"]]
        if len(geometry) != 1:
            raise ValueError(
                f"Expected one geometry for {row.iso3}; found {len(geometry)}"
            )
        geometry = geometry.copy()
        geometry["iso3"] = row.iso3
        frames.append(geometry)

    combined = pd.concat(frames, ignore_index=True)
    return gpd.GeoDataFrame(
        combined, geometry="geometry", crs=frames[0].crs
    )


country_geometries = load_country_geometries(headline)
country_geometries["geometry"] = country_geometries.geometry.simplify(
    tolerance=0.08, preserve_topology=True
)

map_frame = country_geometries.merge(
    headline[
        [
            "iso3", "flood_type", "flood_type_label", "CI",
            "valid_for_ci", "uncertain",
        ]
    ],
    on="iso3",
    how="left",
)
map_frame = gpd.GeoDataFrame(
    map_frame, geometry="geometry", crs=country_geometries.crs
)

cmap = plt.cm.seismic_r
norm = Normalize(vmin=-1, vmax=1)
mpl.rcParams["hatch.linewidth"] = 0.8

fig, axes = plt.subplots(1, 2, figsize=(17, 7), sharex=True, sharey=True)
for ax, flood_type in zip(axes, FLOOD_TYPES):
    subset = map_frame.loc[map_frame["flood_type"].eq(flood_type)]
    valid = subset.loc[subset["valid_for_ci"]]
    uncertain = valid.loc[valid["uncertain"]]

    country_geometries.plot(
        ax=ax, color="#e5e5e5", edgecolor="white", linewidth=0.25
    )
    valid.plot(
        column="CI", ax=ax, cmap=cmap, norm=norm,
        edgecolor="0.45", linewidth=0.25,
    )
    if not uncertain.empty:
        uncertain.plot(
            ax=ax, facecolor="none", edgecolor="0.35",
            linewidth=0.25, hatch="////",
        )

    ax.set_title(
        f"{FLOOD_TYPE_LABELS[flood_type]} flooding (n = {len(valid)})"
    )
    ax.set_xlim(-125, 150)
    ax.set_ylim(-60, 60)
    ax.set_axis_off()

colorbar = fig.colorbar(
    ScalarMappable(norm=norm, cmap=cmap),
    ax=axes, orientation="horizontal", fraction=0.055, pad=0.04,
)
colorbar.set_label("Concentration index (CI)")
colorbar.set_ticks(np.linspace(-1, 1, 5))
fig.legend(
    handles=[
        Patch(
            facecolor="#e5e5e5", edgecolor="white",
            label="No valid CI after filters",
        ),
        Patch(
            facecolor="white", edgecolor="0.35", hatch="////",
            label="Limited observations",
        ),
    ],
    loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.02),
)
fig.suptitle(
    f"National inequality of observed flood exposure, "
    f"{HEADLINE_POPULATION_YEAR} population",
    y=0.98,
)
plt.show()

### 4.2 Ranked national CI distributions

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18, 9), sharey=True)

for ax, flood_type in zip(axes, FLOOD_TYPES):
    plot_data = (
        headline.loc[
            headline["flood_type"].eq(flood_type) & headline["valid_for_ci"]
        ]
        .sort_values("CI")
        .reset_index(drop=True)
    )
    x = np.arange(len(plot_data))
    ax.scatter(
        x, plot_data["CI"],
        color=FLOOD_TYPE_COLORS[flood_type],
        s=35, alpha=0.9, zorder=3,
    )
    ax.axhline(0, color="0.35", linewidth=1, linestyle="--")
    ax.set_xticks(x)
    ax.set_xticklabels(
        [
            f"{row.iso3}{'*' if row.uncertain else ''}"
            for row in plot_data.itertuples()
        ],
        rotation=90,
        fontsize=8,
    )
    ax.set_ylabel("Concentration index (CI)")
    ax.set_title(
        f"{FLOOD_TYPE_LABELS[flood_type]} flooding "
        f"(countries ordered independently; n = {len(plot_data)})"
    )
    ax.set_ylim(-1, 1)
    ax.margins(x=0.01)
    ax.grid(axis="x", visible=False)

axes[-1].set_xlabel("Country (ISO3); * indicates limited flood observations")
fig.suptitle(
    f"National CI for all observed floods using "
    f"{HEADLINE_POPULATION_YEAR} population",
    y=1.01,
)
fig.tight_layout()
plt.show()

### 4.3 Within-country comparison of inland and coastal CI

In [ ]:
ci_comparison = (
    headline.loc[headline["valid_for_ci"]]
    .pivot(
        index=["iso3", "shapeName"],
        columns="flood_type",
        values="CI",
    )
    .dropna(subset=FLOOD_TYPES)
    .reset_index()
)
ci_comparison["absolute_difference"] = (
    ci_comparison["coastal"] - ci_comparison["inland"]
).abs()
labels_to_show = set(
    ci_comparison.nlargest(
        min(10, len(ci_comparison)), "absolute_difference"
    )["iso3"]
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(
    ci_comparison["inland"], ci_comparison["coastal"],
    color="#5e3c99", s=55, alpha=0.8,
)
ax.plot([-1, 1], [-1, 1], color="0.45", linewidth=1, linestyle="--")
ax.axhline(0, color="0.75", linewidth=0.8)
ax.axvline(0, color="0.75", linewidth=0.8)

for row in ci_comparison.itertuples():
    if row.iso3 in labels_to_show:
        ax.annotate(
            row.iso3, (row.inland, row.coastal),
            xytext=(4, 4), textcoords="offset points", fontsize=8,
        )

ax.set(
    xlabel="Inland-flood CI",
    ylabel="Coastal-flood CI",
    xlim=(-1, 1),
    ylim=(-1, 1),
    title=(
        f"Within-country comparison of flood-type CI "
        f"(2010 population; n = {len(ci_comparison)})"
    ),
)
ax.set_aspect("equal", adjustable="box")
plt.show()

display(
    ci_comparison.sort_values(
        "absolute_difference", ascending=False
    ).head(15)
)

## 5. Population growth within observed flood footprints

For each country and flood type, the all-period flood-frequency surface is held fixed while the population snapshot changes. This isolates the population component of cumulative exposure. The comparison metric is:

**exposure growth gap = percentage growth in cumulative exposure − percentage growth in analysed national population**

Positive values mean cumulative exposure grew faster than national population; negative values mean it grew more slowly. Countries with zero exposure in 2000 have undefined percentage growth and are excluded from growth-rate plots.

In [ ]:
temporal = all_metrics.loc[
    all_metrics["Population Coverage (%)"].gt(RWI_COVERAGE_THRESHOLD)
].copy()

baseline = (
    temporal.loc[
        temporal["population_year"].eq(POPULATION_YEARS[0]),
        [
            "iso3", "flood_type", "Population",
            "Total Flood Risk", "exposure_intensity",
        ],
    ]
    .rename(
        columns={
            "Population": "Population_2000",
            "Total Flood Risk": "Total Flood Risk_2000",
            "exposure_intensity": "exposure_intensity_2000",
        }
    )
)

temporal = temporal.merge(baseline, on=["iso3", "flood_type"], how="left")
temporal["population_growth_percent"] = np.where(
    temporal["Population_2000"].gt(0),
    (temporal["Population"] / temporal["Population_2000"] - 1) * 100,
    np.nan,
)
temporal["cumulative_exposure_growth_percent"] = np.where(
    temporal["Total Flood Risk_2000"].gt(0),
    (
        temporal["Total Flood Risk"]
        / temporal["Total Flood Risk_2000"]
        - 1
    ) * 100,
    np.nan,
)
temporal["exposure_growth_gap_pp"] = (
    temporal["cumulative_exposure_growth_percent"]
    - temporal["population_growth_percent"]
)
temporal["exposure_intensity_change_pp"] = (
    temporal["exposure_intensity"] - temporal["exposure_intensity_2000"]
) * 100

growth_2000_2020 = temporal.loc[
    temporal["population_year"].eq(POPULATION_YEARS[-1])
].copy()

display(
    growth_2000_2020[
        [
            "iso3", "shapeName", "flood_type_label",
            "population_growth_percent",
            "cumulative_exposure_growth_percent",
            "exposure_growth_gap_pp",
            "exposure_intensity_change_pp",
        ]
    ].head(10)
)

### 5.1 Aggregate growth across the analysed countries

In [ ]:
aggregate = (
    temporal.groupby(
        ["flood_type", "flood_type_label", "population_year"],
        as_index=False,
    )
    .agg(
        Population=("Population", "sum"),
        cumulative_exposure=("Total Flood Risk", "sum"),
        countries=("iso3", "nunique"),
    )
)

aggregate_baseline = (
    aggregate.loc[
        aggregate["population_year"].eq(POPULATION_YEARS[0]),
        ["flood_type", "Population", "cumulative_exposure"],
    ]
    .rename(
        columns={
            "Population": "Population_2000",
            "cumulative_exposure": "cumulative_exposure_2000",
        }
    )
)
aggregate = aggregate.merge(
    aggregate_baseline, on="flood_type", how="left"
)
aggregate["population_index"] = (
    aggregate["Population"] / aggregate["Population_2000"] * 100
)
aggregate["cumulative_exposure_index"] = (
    aggregate["cumulative_exposure"]
    / aggregate["cumulative_exposure_2000"]
    * 100
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, flood_type in zip(axes, FLOOD_TYPES):
    subset = aggregate.loc[aggregate["flood_type"].eq(flood_type)]
    ax.plot(
        subset["population_year"], subset["population_index"],
        color="0.25", marker="o", linewidth=2,
        label="Analysed population",
    )
    ax.plot(
        subset["population_year"], subset["cumulative_exposure_index"],
        color=FLOOD_TYPE_COLORS[flood_type], marker="o", linewidth=2,
        label="Cumulative population exposure",
    )
    ax.axhline(100, color="0.7", linewidth=0.8)
    ax.set_title(f"{FLOOD_TYPE_LABELS[flood_type]} flooding")
    ax.set_xlabel("Population snapshot year")
    ax.set_xticks(POPULATION_YEARS)
    ax.set_ylabel("Index (2000 = 100)")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
fig.suptitle(
    "Population growth versus growth in cumulative exposure, "
    "fixed 2000–2018 flood-frequency surfaces",
    y=1.05,
)
fig.tight_layout()
plt.show()

display(aggregate)

### 5.2 Country growth comparison, 2000–2020

In [ ]:
plot_growth = growth_2000_2020.loc[
    growth_2000_2020[
        ["population_growth_percent", "cumulative_exposure_growth_percent"]
    ].notna().all(axis=1)
].copy()

fig, axes = plt.subplots(1, 2, figsize=(15, 7), sharex=True, sharey=True)
all_values = plot_growth[
    ["population_growth_percent", "cumulative_exposure_growth_percent"]
].to_numpy()
axis_min = min(-10, np.nanpercentile(all_values, 1) - 5)
axis_max = np.nanpercentile(all_values, 99) + 10

for ax, flood_type in zip(axes, FLOOD_TYPES):
    subset = plot_growth.loc[plot_growth["flood_type"].eq(flood_type)].copy()
    ax.scatter(
        subset["population_growth_percent"],
        subset["cumulative_exposure_growth_percent"],
        s=np.clip(np.sqrt(subset["Total Flood Risk"]) / 4, 20, 250),
        color=FLOOD_TYPE_COLORS[flood_type], alpha=0.65,
        edgecolor="white", linewidth=0.5,
    )
    ax.plot(
        [axis_min, axis_max], [axis_min, axis_max],
        color="0.35", linestyle="--", linewidth=1,
    )

    label_rows = subset.reindex(
        subset["exposure_growth_gap_pp"].abs().nlargest(
            min(8, len(subset))
        ).index
    )
    for row in label_rows.itertuples():
        ax.annotate(
            row.iso3,
            (
                row.population_growth_percent,
                row.cumulative_exposure_growth_percent,
            ),
            xytext=(4, 4), textcoords="offset points", fontsize=8,
        )

    ax.set_title(
        f"{FLOOD_TYPE_LABELS[flood_type]} flooding (n = {len(subset)})"
    )
    ax.set_xlabel("Analysed population growth, 2000–2020 (%)")
    ax.set_ylabel("Cumulative exposure growth, 2000–2020 (%)")
    ax.set_xlim(axis_min, axis_max)
    ax.set_ylim(axis_min, axis_max)

fig.suptitle(
    "Did cumulative exposure grow faster than national population?",
    y=1.01,
)
fig.tight_layout()
plt.show()

In [ ]:
growth_summary = (
    growth_2000_2020.groupby("flood_type_label")
    .agg(
        countries=("iso3", "nunique"),
        comparable_countries=(
            "cumulative_exposure_growth_percent",
            lambda values: values.notna().sum(),
        ),
        median_population_growth_percent=(
            "population_growth_percent", "median"
        ),
        median_exposure_growth_percent=(
            "cumulative_exposure_growth_percent", "median"
        ),
        median_exposure_growth_gap_pp=("exposure_growth_gap_pp", "median"),
        countries_exposure_grew_faster=(
            "exposure_growth_gap_pp", lambda values: values.gt(0).sum()
        ),
        countries_exposure_grew_slower=(
            "exposure_growth_gap_pp", lambda values: values.lt(0).sum()
        ),
    )
)
display(growth_summary)

largest_positive_gaps = (
    growth_2000_2020.loc[
        growth_2000_2020["exposure_growth_gap_pp"].notna(),
        [
            "iso3", "shapeName", "flood_type_label",
            "population_growth_percent",
            "cumulative_exposure_growth_percent",
            "exposure_growth_gap_pp",
        ],
    ]
    .sort_values("exposure_growth_gap_pp", ascending=False)
    .groupby("flood_type_label", group_keys=False)
    .head(10)
)
display(largest_positive_gaps)

### 5.3 Map of the exposure-growth gap

In [ ]:
growth_map = country_geometries.merge(
    growth_2000_2020[
        ["iso3", "flood_type", "exposure_growth_gap_pp"]
    ],
    on="iso3",
    how="left",
)
growth_map = gpd.GeoDataFrame(
    growth_map, geometry="geometry", crs=country_geometries.crs
)

finite_gap = growth_map["exposure_growth_gap_pp"].replace(
    [np.inf, -np.inf], np.nan
)
limit = max(10, np.nanpercentile(np.abs(finite_gap.dropna()), 95))
gap_norm = TwoSlopeNorm(vmin=-limit, vcenter=0, vmax=limit)
gap_cmap = plt.cm.BrBG

fig, axes = plt.subplots(1, 2, figsize=(17, 7), sharex=True, sharey=True)
for ax, flood_type in zip(axes, FLOOD_TYPES):
    subset = growth_map.loc[growth_map["flood_type"].eq(flood_type)]
    valid = subset.loc[subset["exposure_growth_gap_pp"].notna()]

    country_geometries.plot(
        ax=ax, color="#e5e5e5", edgecolor="white", linewidth=0.25
    )
    valid.plot(
        column="exposure_growth_gap_pp",
        ax=ax, cmap=gap_cmap, norm=gap_norm,
        edgecolor="0.45", linewidth=0.25,
    )
    ax.set_title(
        f"{FLOOD_TYPE_LABELS[flood_type]} flooding (n = {len(valid)})"
    )
    ax.set_xlim(-125, 150)
    ax.set_ylim(-60, 60)
    ax.set_axis_off()

colorbar = fig.colorbar(
    ScalarMappable(norm=gap_norm, cmap=gap_cmap),
    ax=axes, orientation="horizontal", fraction=0.055, pad=0.04,
    extend="both",
)
colorbar.set_label(
    "Exposure growth minus national population growth, 2000–2020 "
    "(percentage points)"
)
fig.suptitle(
    "Relative population growth in observed flood footprints",
    y=0.98,
)
plt.show()

## 6. Country explorer

In [ ]:
focus = temporal.loc[temporal["iso3"].eq(FOCUS_ISO3)].copy()
if focus.empty:
    available = ", ".join(sorted(temporal["iso3"].unique()))
    raise ValueError(
        f"FOCUS_ISO3={FOCUS_ISO3!r} not found. Available codes: {available}"
    )

focus_name = focus["shapeName"].iloc[0]
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
specifications = [
    ("Total Flood Risk", "Cumulative population exposure", axes[0, 0]),
    ("exposure_intensity", "Cumulative exposure / population", axes[0, 1]),
    ("CI", "Concentration index (CI)", axes[1, 0]),
    (
        "exposure_growth_gap_pp",
        "Exposure growth gap (percentage points)",
        axes[1, 1],
    ),
]

for metric, ylabel, ax in specifications:
    for flood_type in FLOOD_TYPES:
        subset = focus.loc[
            focus["flood_type"].eq(flood_type)
        ].sort_values("population_year")
        ax.plot(
            subset["population_year"], subset[metric],
            marker="o", linewidth=2,
            color=FLOOD_TYPE_COLORS[flood_type],
            label=FLOOD_TYPE_LABELS[flood_type],
        )
    if metric in {"CI", "exposure_growth_gap_pp"}:
        ax.axhline(0, color="0.45", linewidth=1, linestyle="--")
    if metric == "Total Flood Risk":
        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda value, position: f"{value / 1_000_000:.1f}m"
            )
        )
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Population snapshot year")
    ax.set_xticks(POPULATION_YEARS)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
fig.suptitle(
    f"{focus_name} ({FOCUS_ISO3}): population sensitivity "
    "of all-period metrics",
    y=1.02,
)
fig.tight_layout()
plt.show()

display(
    focus[
        [
            "flood_type_label", "population_year", "Population",
            "Total Flood Risk", "exposure_intensity", "CI",
            "population_growth_percent",
            "cumulative_exposure_growth_percent",
            "exposure_growth_gap_pp",
        ]
    ].sort_values(["flood_type_label", "population_year"])
)

## 7. Analysis-ready tables

The following objects remain available for follow-on analysis or export:

- `headline`: one 2010-population row per country and flood type, including filters and uncertainty flags.
- `ci_comparison`: countries with valid inland and coastal CI values.
- `temporal`: all five population snapshots with population and cumulative-exposure growth measures.
- `growth_2000_2020`: one 2000–2020 comparison per country and flood type.
- `growth_summary`: compact flood-type summary.

No files are exported automatically. This keeps the notebook read-only; use the optional cell below if a CSV export is wanted.

In [ ]:
# Optional exports (uncomment if needed).
# output_dir = (
#     PROJECT_ROOT / "data" / "results" / "social_flood" / "summary_tables"
# )
# output_dir.mkdir(parents=True, exist_ok=True)
# headline.drop(columns=["source_path"], errors="ignore").to_csv(
#     output_dir / "observed_national_CI_2010_population.csv", index=False
# )
# growth_2000_2020.drop(columns=["source_path"], errors="ignore").to_csv(
#     output_dir / "observed_population_exposure_growth_2000_2020.csv",
#     index=False,
# )

## Interpretation checklist

- CI is a relative inequality measure within each country; it does not measure absolute exposure.
- Coastal CI is undefined for countries with no recorded coastal exposure, so coastal sample sizes are smaller.
- The 2010 cross-section holds population year constant while comparing the full 2000–2018 flood-frequency surfaces.
- The temporal section holds the flood-frequency surface constant while changing population snapshots. It therefore isolates the population component, not changes in flood occurrence.
- `Total Flood Risk` is cumulative person-event exposure. Do not interpret it as unique residents or subtract it from population.
- The exposure-growth gap is descriptive. It can reflect spatially uneven population change within the RWI-covered area and does not establish causality.
- Exact in/out-of-floodplain population growth requires binary flood-footprint outputs in addition to these frequency-weighted metrics.